# 3D-Visualisierung: Lineare Regression vs. Entscheidungsbaum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_16/3d_simple_comparison.ipynb)

## Einfaches Beispiel: Wie sehen die Algorithmen den 3D-Raum?

**Lineare Regression:** Glatte Ebene  
**Entscheidungsbaum:** Ein Quader pro Blatt, mit einem Vorhersageplateau darin

Hier sagen beide Modelle eine **Zahl** voraus, keine Klasse wie eine Blumenart. Das nennt man Regression.

Schauen wir uns das visuell an!

In [ ]:
# Einfache Bibliotheken
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

print("📦 Bibliotheken geladen!")

## Schritt 1: Einfache Daten erstellen

Wir erstellen 50 Punkte mit 2 Features (x, y) und einem Target (z).

In [ ]:
# Einfache Daten erstellen
np.random.seed(42)

# 50 Punkte
x = np.random.uniform(-2, 2, 50)
y = np.random.uniform(-2, 2, 50)

# Einfacher linearer Zusammenhang: z = x + y + etwas Rauschen
z = x + y + np.random.normal(0, 0.3, 50)

# Beide Messungen als gemeinsame Tabelle für sklearn
X = np.column_stack([x, y])

print(f"✅ {len(x)} Datenpunkte erstellt")
print(f"📊 Features: x und y")
print(f"🎯 Target: z")

## Schritt 2: Beide Modelle trainieren

In [ ]:
# Lineare Regression
lr_model = LinearRegression()
lr_model.fit(X, z)

# Entscheidungsbaum (einfach gehalten)
tree_model = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_model.fit(X, z)

# Hilfsfunktion für die Zeichnung; den Code müsst ihr nicht im Detail verstehen.
def get_leaf_regions(tree, X_train, z_train):
    """Liefert Blattbereiche, Trainingshöhen und Vorhersagen im gezeichneten Ausschnitt."""
    tree_ = tree.tree_
    leaf_ids = tree.apply(X_train)
    regions = []

    def recurse(node, x_min, x_max, y_min, y_max):
        if tree_.children_left[node] != tree_.children_right[node]:
            feature = tree_.feature[node]
            threshold = tree_.threshold[node]
            if feature == 0:
                recurse(tree_.children_left[node], x_min, threshold, y_min, y_max)
                recurse(tree_.children_right[node], threshold, x_max, y_min, y_max)
            else:
                recurse(tree_.children_left[node], x_min, x_max, y_min, threshold)
                recurse(tree_.children_right[node], x_min, x_max, threshold, y_max)
        else:
            z_values = z_train[leaf_ids == node]
            regions.append({
                'leaf_id': node,
                'x_min': x_min, 'x_max': x_max,
                'y_min': y_min, 'y_max': y_max,
                'z_min': float(z_values.min()), 'z_max': float(z_values.max()),
                'samples': len(z_values),
                'prediction': tree_.value[node][0][0]
            })

    recurse(0, -2, 2, -2, 2)
    return regions

regions = get_leaf_regions(tree_model, X, z)

def get_box_vertices(region):
    """Die gleichen Quaderecken für beide Zeichnungen."""
    x_min, x_max = region['x_min'], region['x_max']
    y_min, y_max = region['y_min'], region['y_max']
    z_min, z_max = region['z_min'], region['z_max']
    return [
        [x_min, y_min, z_min], [x_max, y_min, z_min],
        [x_max, y_max, z_min], [x_min, y_max, z_min],
        [x_min, y_min, z_max], [x_max, y_min, z_max],
        [x_max, y_max, z_max], [x_min, y_max, z_max]
    ]

BOX_EDGES = [(0, 1), (1, 2), (2, 3), (3, 0),
             (4, 5), (5, 6), (6, 7), (7, 4),
             (0, 4), (1, 5), (2, 6), (3, 7)]

print("🤖 Beide Modelle trainiert!")
print(f"📈 Lineare Regression: z = {lr_model.coef_[0]:.2f}*x + {lr_model.coef_[1]:.2f}*y + {lr_model.intercept_:.2f}")
print(f"🌳 Entscheidungsbaum: {tree_model.get_n_leaves()} Blätter, Tiefe {tree_model.get_depth()}")

## Schritt 3: 3D-Visualisierung

Die waagerechten Achsen zeigen x und y. Die Höhe zeigt z: bei den roten Punkten den Trainingswert, beim Plateau die Vorhersage. Jeder Quader steht für ein Blatt. Er reicht vom kleinsten bis zum größten Trainings-z-Wert in diesem Blatt. Das schwarze Plateau zeigt ihren Mittelwert, den der Baum auch für neue Punkte in diesem x-y-Bereich vorhersagt. Der Zeichencode ist eine fertige Hilfe.

In [ ]:
# Raster für Vorhersagen erstellen
x_range = np.linspace(-2, 2, 20)
y_range = np.linspace(-2, 2, 20)
X_grid, Y_grid = np.meshgrid(x_range, y_range)

# Vorhersagen für das gesamte Raster
grid_points = np.column_stack([X_grid.ravel(), Y_grid.ravel()])
Z_linear = lr_model.predict(grid_points).reshape(X_grid.shape)
Z_tree = tree_model.predict(grid_points).reshape(X_grid.shape)

print("🌐 Vorhersage-Oberflächen berechnet!")

In [ ]:
# 3D-Plot erstellen
fig = plt.figure(figsize=(16, 6))

# Lineare Regression
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(x, y, z, color='red', s=20, alpha=0.8, label='Datenpunkte')  # kleinere Punkte
ax1.plot_surface(X_grid, Y_grid, Z_linear, alpha=0.3, color='blue')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')
ax1.set_title('Lineare Regression\n(Glatte Ebene)', fontweight='bold')
ax1.legend()

# Entscheidungsbaum: Trainingsquader und Mittelwert pro Blatt
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(x, y, z, color='red', s=20, alpha=0.8, label='Datenpunkte')  # kleinere Punkte

for region in regions:
    vertices = get_box_vertices(region)
    faces = [[vertices[i] for i in indices] for indices in
             [[0, 3, 2, 1], [4, 5, 6, 7], [0, 1, 5, 4],
              [1, 2, 6, 5], [2, 3, 7, 6], [3, 0, 4, 7]]]
    ax2.add_collection3d(Poly3DCollection(faces, facecolors='orange',
                                        edgecolors='orange', linewidths=0.5, alpha=0.12))
    for start, end in BOX_EDGES:
        ax2.plot(*np.array([vertices[start], vertices[end]]).T,
                 color='darkorange', linewidth=0.8)
    xs = np.array([[region['x_min'], region['x_max']]] * 2)
    ys = np.array([[region['y_min']] * 2, [region['y_max']] * 2])
    zs = np.full((2, 2), region['prediction'])
    ax2.plot_surface(xs, ys, zs, alpha=0.5, color='black',
                     edgecolor='black', linewidth=0.5, antialiased=False)

ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')
ax2.set_title('Entscheidungsbaum\n(Trainingsquader und Mittelwerte)', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

print("🎨 3D-Visualisierung fertig!")
print("💡 Jeder Quader steht für ein Blatt und umfasst dessen Trainingspunkte.")
print("Das schwarze Plateau zeigt den Mittelwert und damit die Vorhersage des Blatts.")

## Was sehen wir?

**🔵 Lineare Regression (links):**
- Erzeugt eine **glatte, schräge Ebene**
- Die Ebene geht durch die Punktwolke
- Die gelernte Formel steht in der Ausgabe oben; sie liegt nahe an z = x + y.

**🟡 Entscheidungsbaum (rechts):**
- **Ein Quader steht für ein Blatt** des trainierten Baums.
- Seine Grundfläche ist der x-y-Bereich des Blatts.
- Seine Höhe reicht vom kleinsten bis zum größten z-Wert der Trainingspunkte darin.
- Das **schwarze Plateau** liegt auf ihrem Mittelwert: der Vorhersage des Blatts.
- Neue Punkte bekommen anhand von x und y einen Bereich und dessen Mittelwert zugewiesen.
- Ihr tatsächlicher z-Wert kann auch außerhalb der bisherigen Trainingshöhen liegen.
- Hat ein Blatt nur einen Trainings-z-Wert, ist sein Quader flach.
- **Wichtig:** Grenzen verlaufen parallel zu x- und y-Achsen!
- Der 2D-Raum wird in rechteckige Bereiche aufgeteilt

## Warum achsparallele Grenzen?

Der Entscheidungsbaum teilt mit Regeln wie:
- "Wenn x < 0.5, dann..."
- "Wenn y > -1.2, dann..."

Diese Regeln erzeugen **senkrechte und waagerechte Linien** im 2D-Raum, die sich zu **rechteckigen Bereichen** kombinieren. Jeder Bereich bekommt eine konstante Vorhersage → flache Plateaus!

## Fazit

- **Linearer Zusammenhang** → Eine Ebene kann gut passen, wie in diesem Beispiel.
- **Stufenförmiger Zusammenhang** → Ein Entscheidungsbaum kann gut passen.
- Welche Vorhersagen wirklich besser sind, müsste man an neuen Daten prüfen.

Das ist der Grundunterschied! 🎯

## 🎮 Bonus: Interaktive 3D-Plots

Die gleichen Trainingsquader, aber **interaktiv**! Jeder Quader steht für ein Blatt. Die schwarze Fläche darin zeigt den Mittelwert, den der Baum vorhersagt. Der Zeichencode ist kein zusätzlicher Lernstoff. Du kannst:
- **Drehen** mit der Maus
- **Zoomen** mit dem Mausrad
- **Verschieben** durch Klicken und Ziehen

In [ ]:
# Plotly für interaktive Plots
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("🚀 Plotly geladen - bereit für interaktive Plots!")

In [ ]:
# Funktion für echte 3D-Quader
def create_3d_box(region, color='orange', opacity=0.3):
    """Erstellt einen vollständigen 3D-Quader mit go.Mesh3d"""
    vertices = get_box_vertices(region)
    
    x_coords = [v[0] for v in vertices]
    y_coords = [v[1] for v in vertices]
    z_coords = [v[2] for v in vertices]
    
    # Die Reihenfolge richtet die Außenseiten der Flächen nach außen.
    triangles = [
        [0,2,1], [0,3,2],  # unten
        [4,6,7], [4,5,6],  # oben
        [0,5,4], [0,1,5],  # vorne
        [2,7,6], [2,3,7],  # hinten
        [0,7,3], [0,4,7],  # links
        [1,6,5], [1,2,6]   # rechts
    ]
    
    i_coords = [t[0] for t in triangles]
    j_coords = [t[1] for t in triangles]
    k_coords = [t[2] for t in triangles]
    
    return go.Mesh3d(
        x=x_coords, y=y_coords, z=z_coords,
        i=i_coords, j=j_coords, k=k_coords,
        color=color, opacity=opacity,
        showscale=False
    )

# Interaktive 3D-Plots erstellen
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Lineare Regression", "Ein Trainingsquader pro Blatt"],
    specs=[[{"type": "scene"}, {"type": "scene"}]]
)

# Datenpunkte (für beide Plots)
scatter_trace = go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers',
    marker=dict(size=4, color='red', opacity=0.9),
    name='Datenpunkte'
)

# Lineare Regression - feineres Grid für glatte Oberfläche
x_range_fine = np.linspace(-2, 2, 30)
y_range_fine = np.linspace(-2, 2, 30)
X_grid_fine, Y_grid_fine = np.meshgrid(x_range_fine, y_range_fine)
grid_points_fine = np.column_stack([X_grid_fine.ravel(), Y_grid_fine.ravel()])
Z_linear_fine = lr_model.predict(grid_points_fine).reshape(X_grid_fine.shape)

linear_surface = go.Surface(
    x=X_grid_fine, y=Y_grid_fine, z=Z_linear_fine,
    colorscale='Blues', opacity=0.5,
    name='Lineare Regression',
    showscale=False
)

# Die Blattbereiche wurden oben auch für den statischen Plot verwendet.
print(f"🔍 Entscheidungsbaum hat {len(regions)} rechteckige Bereiche")

# Erstelle 3D-Quader für jeden Bereich
tree_boxes = []
colors = ['orange', 'gold', 'coral', 'sandybrown', 'darkorange', 'peachpuff', 'papayawhip', 'moccasin']

for i, region in enumerate(regions):
    box = create_3d_box(
        region,
        color=colors[i % len(colors)],
        opacity=0.15
    )
    tree_boxes.append(box)

# Plots hinzufügen
# Linke Seite: Lineare Regression
fig.add_trace(scatter_trace, row=1, col=1)
fig.add_trace(linear_surface, row=1, col=1)

# Rechte Seite: Entscheidungsbaum mit echten 3D-Quadern
scatter_trace_2 = go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers',
    marker=dict(size=5, color='red', opacity=1.0),
    name='Datenpunkte'
)
fig.add_trace(scatter_trace_2, row=1, col=2)

# Füge alle 3D-Quader hinzu
for i, box in enumerate(tree_boxes):
    box.name = f'Blatt {i+1}: {regions[i]["samples"]} Trainingspunkte'
    fig.add_trace(box, row=1, col=2)

# Alle zwölf Außenkanten machen jeden Quader als vollständigen Körper sichtbar.
for i, region in enumerate(regions):
    vertices = get_box_vertices(region)
    edge_x, edge_y, edge_z = [], [], []
    for start, end in BOX_EDGES:
        for vertex in (vertices[start], vertices[end]):
            edge_x.append(vertex[0])
            edge_y.append(vertex[1])
            edge_z.append(vertex[2])
        edge_x.append(None)
        edge_y.append(None)
        edge_z.append(None)
    fig.add_trace(go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z, mode='lines',
        line=dict(color=colors[i % len(colors)], width=3),
        showlegend=False, name=f'Außenkanten Blatt {i+1}'
    ), row=1, col=2)

# Schwarze Plateaus und Rahmen markieren die Vorhersagehöhe.
for i, region in enumerate(regions):
    z_val = region['prediction']
    fig.add_trace(go.Surface(
        x=[[region['x_min'], region['x_max']]] * 2,
        y=[[region['y_min']] * 2, [region['y_max']] * 2],
        z=[[z_val] * 2] * 2,
        colorscale=[[0, 'black'], [1, 'black']], opacity=0.5,
        showscale=False, name=f'Mittelwert Blatt {i+1}'
    ), row=1, col=2)
    
    # Rechteckiger Rahmen auf der Höhe der Vorhersage
    frame_x = [region['x_min'], region['x_max'], region['x_max'], region['x_min'], region['x_min']]
    frame_y = [region['y_min'], region['y_min'], region['y_max'], region['y_max'], region['y_min']]
    frame_z = [z_val] * 5
    
    fig.add_trace(go.Scatter3d(
        x=frame_x, y=frame_y, z=frame_z,
        mode='lines',
        line=dict(color='black', width=4),
        showlegend=False,
        name=f'Rahmen {i+1}'
    ), row=1, col=2)

# Layout anpassen
fig.update_layout(
    title="🎮 Interaktive 3D-Visualisierung (Drehen & Zoomen möglich!)",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z"
    ),
    scene2=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z"
    ),
    height=600,
    showlegend=False
)

fig.show()

print("🎯 Interaktive Trainingsquader bereit!")
print("💡 Tipp: Klicke und ziehe zum Drehen, Mausrad zum Zoomen!")
print(f"📦 Der Baum teilt die x-y-Fläche in {len(regions)} rechteckige Bereiche.")
print("🔍 Die x- und y-Werte bestimmen den Bereich und damit die Vorhersage für z.")
print("Die tatsächliche Höhe eines Datenpunkts kann von dieser Vorhersage abweichen.")
print("Die Quader umfassen die Trainingswerte je Blatt, nicht unbedingt die Werte neuer Punkte.")
print("⚫ Schwarze Plateaus und Rahmen zeigen den Mittelwert je Blatt.")

# Zeige Details der Bereiche
for i, region in enumerate(regions):
    print(f"Blatt {i+1} ({region['samples']} Trainingspunkte): "
          f"x∈[{region['x_min']:.2f}, {region['x_max']:.2f}], "
          f"y∈[{region['y_min']:.2f}, {region['y_max']:.2f}], "
          f"Trainings-z∈[{region['z_min']:.2f}, {region['z_max']:.2f}] "
          f"→ Mittelwert/Vorhersage={region['prediction']:.2f}")

## 🎯 Perfekt!

Jetzt hast du beide Visualisierungen:
- **Statische Plots** (matplotlib) - für Screenshots und Dokumentation
- **Interaktive Plots** (plotly) - zum Erkunden und Verstehen

**Probiere es aus:**
1. **Klicke und ziehe** um die 3D-Ansicht zu drehen
2. **Mausrad** zum Rein- und Rauszoomen  
3. **Doppelklick** um zur ursprünglichen Ansicht zurückzukehren

So siehst du noch besser, wie die **glatte Ebene** der linearen Regression sich von den **stufigen Plateaus** des Entscheidungsbaums unterscheidet!